# Reinterpret one Gravlax archive with two annotations

This notebook compares two gene annotations against the **same rooted molecular-evidence archive**. It downloads only manifest-pinned bytes, verifies every SHA-256 digest, and executes Gravlax; it contains no cached scientific result. Its final figure is a deterministic SVG built from the typed result table using only the Python standard library and the IPython display API supplied by Colab.

In [ ]:
#@title Immutable demo manifest (required)
MANIFEST_URL = "https://github.com/COMBINE-lab/gravlax/releases/download/demo-data-v1/demo-manifest.json" #@param {type:"string"}
MANIFEST_SHA256 = "82c34aad442d478f1cb1243a6ccfe8ad9f937b81d9e1f946a8eb2cfc498214fd" #@param {type:"string"}

if not MANIFEST_URL or not MANIFEST_SHA256:
    raise RuntimeError("Demo capsule not published/configured: set the immutable manifest URL and its SHA-256. No fallback URL is used.")

In [ ]:
import hashlib, json, os, re, subprocess, sys, tarfile, urllib.parse, urllib.request, zipfile
from pathlib import Path

WORK = Path('/content/gravlax-demo')
WORK.mkdir(parents=True, exist_ok=True)
HEX64 = re.compile(r'^[0-9a-f]{64}$')

def download_verified(url, sha256, destination):
    parsed_url = urllib.parse.urlsplit(url) if isinstance(url, str) else None
    if parsed_url is None or parsed_url.scheme != 'https' or not parsed_url.netloc:
        raise ValueError(f'asset URL must be HTTPS, got {url!r}')
    if 'latest' in (segment.lower() for segment in parsed_url.path.split('/')):
        raise ValueError(f'asset URL must be immutable; /latest/ is not allowed: {url!r}')
    if not isinstance(sha256, str) or not HEX64.fullmatch(sha256):
        raise ValueError('asset SHA-256 must be 64 lowercase hexadecimal characters')
    destination = Path(destination)
    temporary = destination.with_suffix(destination.suffix + '.part')
    digest = hashlib.sha256()
    with urllib.request.urlopen(url) as source, temporary.open('wb') as sink:
        while block := source.read(1 << 20):
            digest.update(block); sink.write(block)
    if digest.hexdigest() != sha256:
        temporary.unlink(missing_ok=True)
        raise RuntimeError(f'SHA-256 mismatch for {url}')
    temporary.replace(destination)
    return destination

manifest_path = download_verified(MANIFEST_URL, MANIFEST_SHA256, WORK / 'manifest.json')
manifest = json.loads(manifest_path.read_text())
if manifest.get('schema') != 'gravlax.demo-capsule.v1':
    raise RuntimeError('unsupported or missing demo manifest schema')
for section in ('software', 'resources', 'stories'):
    if not isinstance(manifest.get(section), dict): raise RuntimeError(f'manifest {section} must be an object')
required_software_fields = {'version', 'aie', 'python_wheel'}
if required_software_fields.difference(manifest['software']): raise RuntimeError(f'manifest software lacks {sorted(required_software_fields.difference(manifest["software"]))}')

RESERVED_ASSET_FILENAMES = {'.', '..', 'manifest.json', 'event-discovery.aicollection', 'junction-drilldown.aicollection'}
asset_filenames = {}
for asset_name, asset_spec in [('software.aie', manifest['software'].get('aie')), ('software.python_wheel', manifest['software'].get('python_wheel')), *[(f'resources.{name}', spec) for name, spec in manifest['resources'].items()]]:
    if not isinstance(asset_spec, dict): raise RuntimeError(f'asset declaration {asset_name} must be an object')
    filename = asset_spec.get('filename')
    if not isinstance(filename, str) or not filename or Path(filename).name != filename or filename in RESERVED_ASSET_FILENAMES:
        raise ValueError(f'asset declaration {asset_name} has an invalid or reserved filename: {filename!r}')
    if filename in asset_filenames: raise ValueError(f'duplicate asset filename {filename!r}: {asset_filenames[filename]} and {asset_name}')
    asset_filenames[filename] = asset_name

def fetch(spec):
    required = ('url', 'sha256', 'filename')
    if not isinstance(spec, dict) or any(not spec.get(key) for key in required):
        raise RuntimeError(f'incomplete published asset declaration: {spec!r}')
    if Path(spec['filename']).name != spec['filename']:
        raise ValueError('asset filename must be a basename')
    return download_verified(spec['url'], spec['sha256'], WORK / spec['filename'])

def install_tools():
    wheel = fetch(manifest['software']['python_wheel'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--force-reinstall', '--no-deps', str(wheel)], check=True)
    spec = manifest['software']['aie']; bundle = fetch(spec)
    member = spec.get('member')
    if member:
        payload = None
        if zipfile.is_zipfile(bundle):
            with zipfile.ZipFile(bundle) as archive: payload = archive.read(member)
        else:
            with tarfile.open(bundle, 'r:*') as archive:
                item = archive.getmember(member)
                if not item.isfile(): raise RuntimeError('configured aie archive member is not a file')
                payload = archive.extractfile(item).read()
        binary = WORK / 'aie'; binary.write_bytes(payload)
    else:
        binary = bundle
    binary.chmod(0o755)
    return binary

AIE = install_tools()
from gravlax import Client, __version__ as PYTHON_VERSION
EXPECTED_VERSION = manifest['software'].get('version')
if not isinstance(EXPECTED_VERSION, str) or not EXPECTED_VERSION: raise RuntimeError('manifest software.version must be nonempty')
CLI_VERSION = subprocess.run([str(AIE), '--version'], check=True, capture_output=True, text=True).stdout.strip()
if CLI_VERSION != f'aie {EXPECTED_VERSION}': raise RuntimeError(f'CLI version mismatch: {CLI_VERSION!r} != aie {EXPECTED_VERSION}')
if PYTHON_VERSION != EXPECTED_VERSION: raise RuntimeError(f'Python version mismatch: {PYTHON_VERSION!r} != {EXPECTED_VERSION}')
client = Client(binary=AIE)
print(CLI_VERSION)

In [ ]:
story = manifest['stories'].get('annotation_reinterpretation')
required_story_fields = {'archive', 'annotation_before', 'annotation_after', 'assembly', 'before_label', 'after_label', 'expected_gene_id', 'expected_min_signed_delta'}
if not isinstance(story, dict) or required_story_fields.difference(story): raise RuntimeError(f'annotation_reinterpretation story lacks {sorted(required_story_fields.difference(story or {}))}')
resources = manifest['resources']
def story_resource(field):
    name = story[field]; spec = resources.get(name)
    if not isinstance(name, str) or not isinstance(spec, dict): raise RuntimeError(f'story resource {field} is not declared: {name!r}')
    return spec
archive_spec = story_resource('archive')
archive = fetch(archive_spec)
before = fetch(story_resource('annotation_before'))
after = fetch(story_resource('annotation_after'))

expected_root = archive_spec.get('archive_root')
if not expected_root: raise RuntimeError('archive lacks a rooted identity in the manifest')
identity = client.result_raw(['inspect-archive', archive, '--json'])['native_identity']
observed_root = f"{identity['scheme']}:{identity['blake3']}"
if observed_root != expected_root: raise RuntimeError(f'archive root mismatch: {observed_root}')

comparison = client.compare_annotations(
    archive, before, after,
    assembly=story['assembly'],
    annotation_a_label=story['before_label'],
    annotation_b_label=story['after_label'],
    max_molecule_witnesses=story.get('max_molecule_witnesses', 10000),
)
print(json.dumps(dict(comparison.summary), indent=2))
deltas = comparison.count_deltas.records()
locked_rows = [row for row in deltas if row.get('comparison_gene_id') == story['expected_gene_id']]
if not locked_rows: raise RuntimeError(f"locked gene is absent from count deltas: {story['expected_gene_id']}")
locked_delta = sum(row['signed_delta_b_minus_a'] for row in locked_rows)
if locked_delta < story['expected_min_signed_delta']: raise RuntimeError(f"locked gene delta {locked_delta} is below {story['expected_min_signed_delta']}")
print(f"verified {story['expected_gene_id']} signed UMI delta: {locked_delta:+d}")
for row in sorted(deltas, key=lambda row: abs(row['signed_delta_b_minus_a']), reverse=True)[:20]:
    print(row)

In [ ]:
# Deterministic SVG generated only from the typed count-delta table.
from html import escape
from IPython.display import SVG, display

def require_table_columns(table, expected, label):
    missing = set(expected).difference(table.columns)
    if missing: raise RuntimeError(f'{label} table is missing columns: {sorted(missing)}')

def signed_bar_svg(items, title, value_label):
    width, label_width, plot_width, row_height = 920, 290, 520, 28
    items = items[:15]
    height = 92 + row_height * max(1, len(items))
    zero = label_width + plot_width / 2
    peak = max((abs(value) for _, value in items), default=1) or 1
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" role="img" aria-label="{escape(title)}">',
             '<style>text{font-family:system-ui,sans-serif;font-size:12px}.title{font-size:16px;font-weight:600}.axis{fill:#5b6470}</style>',
             f'<text class="title" x="8" y="22">{escape(title)}</text>',
             f'<text class="axis" x="{zero}" y="43" text-anchor="middle">0 · {escape(value_label)}</text>',
             f'<line x1="{zero}" x2="{zero}" y1="50" y2="{height - 12}" stroke="#7b8490"/>']
    if not items:
        parts.append('<text x="8" y="72">No count deltas were returned for this capsule.</text>')
    for index, (label, value) in enumerate(items):
        y = 58 + index * row_height; span = abs(value) * (plot_width / 2 - 8) / peak
        x = zero if value >= 0 else zero - span; color = '#177e89' if value >= 0 else '#d95f02'
        parts.extend([f'<text x="{label_width - 8}" y="{y + 14}" text-anchor="end">{escape(str(label)[:42])}</text>',
                      f'<rect x="{x}" y="{y}" width="{max(span, 1)}" height="18" rx="2" fill="{color}"/>',
                      f'<text x="{zero + (span + 5 if value >= 0 else -span - 5)}" y="{y + 14}" text-anchor="{"start" if value >= 0 else "end"}">{value:+d}</text>'])
    parts.append('</svg>'); return ''.join(parts)

delta_table = comparison.count_deltas
require_table_columns(delta_table, {'comparison_gene_id', 'signed_delta_b_minus_a'}, 'count_deltas')
gene_delta = {}
for row in delta_table.records():
    gene = row['comparison_gene_id']; gene_delta[gene] = gene_delta.get(gene, 0) + row['signed_delta_b_minus_a']
plot_rows = sorted(((gene, delta) for gene, delta in gene_delta.items() if delta), key=lambda item: (-abs(item[1]), item[0]))
display(SVG(data=signed_bar_svg(plot_rows, f'Annotation reinterpretation: {story["before_label"]} → {story["after_label"]}', 'summed UMI count change')))